# Noise models

反復測定から観測分散を推定する `ReplicateNoiseSingleTaskGP` を使い、latent function の不確かさと測定ノイズを分離します。


## 反復測定データ


In [ ]:
import torch
from robotorchan.models import ReplicateNoiseSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
base_X = torch.tensor([[0.0], [0.5], [1.0]])
X = base_X.repeat_interleave(3, dim=0)
Y = torch.sin(2 * torch.pi * X) + 0.05 * torch.randn_like(X)


## 反復を集約してGPを構築


In [ ]:
model = ReplicateNoiseSingleTaskGP.from_replicates(X, Y)
model.fit()
print(model.raw_train_X)
print(model.raw_train_Yvar)
print(model.replicate_counts)


## Posterior


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
posterior = model.posterior(test_X)
print(posterior.mean.squeeze(-1))


## Replicate-aware candidate proposal

反復観測から推定したノイズ分散を持つモデルで、予測平均最大の候補を選択します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.sin(2 * torch.pi * candidate)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)


## Mixed replicate-aware candidate proposal

連続値とカテゴリ値の組合せごとに反復測定を集約し、元の混合入力空間で予測平均最大の候補を選びます。これは獲得関数の最適化ではありません。


In [ ]:
from robotorchan.models import MixedReplicateNoiseSingleTaskGP

mixed_base_X = torch.tensor(
    [[0.0, 0.0], [0.5, 0.0], [1.0, 0.0],
     [0.0, 1.0], [0.5, 1.0], [1.0, 1.0]]
)
mixed_replicate_X = mixed_base_X.repeat_interleave(3, dim=0)
mixed_replicate_Y = (
    torch.sin(2 * torch.pi * mixed_replicate_X[:, :1])
    + 0.2 * mixed_replicate_X[:, 1:2]
    + 0.03 * torch.randn(mixed_replicate_X.shape[0], 1)
)
mixed_model = MixedReplicateNoiseSingleTaskGP.from_replicates(
    mixed_replicate_X, mixed_replicate_Y, cat_dims=[1]
)
mixed_model.fit()
mixed_pool = torch.cartesian_prod(
    torch.linspace(0, 1, 21), torch.tensor([0.0, 1.0])
)
mixed_model.eval()
with torch.no_grad():
    mixed_scores = mixed_model.posterior(mixed_pool).mean.squeeze(-1)
mixed_candidate = mixed_pool[mixed_scores.argmax()].unsqueeze(0)
mixed_observation = (
    torch.sin(2 * torch.pi * mixed_candidate[:, :1])
    + 0.2 * mixed_candidate[:, 1:2]
)
assert mixed_candidate.shape == (1, 2)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
print("mixed candidate:", mixed_candidate)
print("evaluated objective:", mixed_observation)


## Input-dependent noise: single-task GPs

既存の `fit_heteroskedastic(iterations=1)` で平均モデルと対数ノイズモデルを交互学習します。平均のPosteriorと予測観測分散を分けて確認し、有限プールから候補を選びます。


In [ ]:
from robotorchan.models import (
    HeteroskedasticSingleTaskGP,
    MixedHeteroskedasticSingleTaskGP,
)

hetero_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
hetero_Y = (
    torch.sin(2 * torch.pi * hetero_X)
    + 0.03 * (1 + hetero_X) * torch.randn_like(hetero_X)
)
hetero_model = HeteroskedasticSingleTaskGP(hetero_X, hetero_Y)
hetero_model.fit_heteroskedastic(iterations=1)
hetero_model.eval()
hetero_pool = torch.linspace(0, 1, 31).unsqueeze(-1)
with torch.no_grad():
    hetero_scores = hetero_model.posterior(hetero_pool).mean.squeeze(-1)
    hetero_noise = hetero_model.predicted_noise(hetero_pool)
hetero_candidate = hetero_pool[hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(hetero_noise).all()
print("heteroskedastic candidate:", hetero_candidate)
print("objective:", torch.sin(2 * torch.pi * hetero_candidate))

mixed_hetero_X = torch.cat(
    [hetero_X, torch.arange(10).remainder(2).to(hetero_X).unsqueeze(-1)],
    dim=-1,
)
mixed_hetero_Y = hetero_Y + 0.2 * mixed_hetero_X[:, 1:2]
mixed_hetero_model = MixedHeteroskedasticSingleTaskGP(
    mixed_hetero_X, mixed_hetero_Y, cat_dims=[1]
)
mixed_hetero_model.fit_heteroskedastic(iterations=1)
mixed_hetero_model.eval()
with torch.no_grad():
    mixed_hetero_scores = (
        mixed_hetero_model.posterior(mixed_pool).mean.squeeze(-1)
    )
    mixed_hetero_noise = mixed_hetero_model.predicted_noise(mixed_pool)
mixed_hetero_candidate = mixed_pool[mixed_hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(mixed_hetero_noise).all()
print("mixed heteroskedastic candidate:", mixed_hetero_candidate)
print(
    "objective:",
    torch.sin(2 * torch.pi * mixed_hetero_candidate[:, :1])
    + 0.2 * mixed_hetero_candidate[:, 1:2],
)


## Heteroskedastic multitask and multi-fidelity GPs

Long-formatのタスク列と忠実度列を区別して入力依存ノイズを学習します。候補は対象タスクまたは高忠実度に固定した有限プールの予測平均から選択します。


In [ ]:
from robotorchan.models import (
    HeteroskedasticMultiFidelityGP,
    HeteroskedasticMultiTaskGP,
    MixedHeteroskedasticMultiTaskGP,
)

task0_X = torch.cat([hetero_X, torch.zeros_like(hetero_X)], dim=-1)
task1_X = torch.cat([hetero_X, torch.ones_like(hetero_X)], dim=-1)
hetero_task_X = torch.cat([task0_X, task1_X], dim=0)
hetero_task_Y = torch.cat([hetero_Y, hetero_Y + 0.2], dim=0)
hetero_task_pool = torch.cat(
    [hetero_pool, torch.zeros_like(hetero_pool)], dim=-1
)
task_noise_model = HeteroskedasticMultiTaskGP(
    hetero_task_X, hetero_task_Y, task_feature=1
)
task_noise_model.fit_heteroskedastic(iterations=1)
task_noise_model.eval()
with torch.no_grad():
    task_scores = task_noise_model.posterior(hetero_task_pool).mean.reshape(-1)
    task_noise = task_noise_model.predicted_noise(hetero_task_pool)
task_candidate = hetero_task_pool[task_scores.argmax()].unsqueeze(0)
assert torch.isfinite(task_noise).all()
print("task candidate:", task_candidate)
print("task 0 objective:", torch.sin(2 * torch.pi * task_candidate[:, :1]))

mixed_task_X = torch.cat(
    [mixed_hetero_X, torch.zeros(mixed_hetero_X.shape[0], 1)], dim=-1
)
mixed_task_X = torch.cat(
    [
        mixed_task_X,
        torch.cat(
            [mixed_hetero_X, torch.ones(mixed_hetero_X.shape[0], 1)], dim=-1
        ),
    ],
    dim=0,
)
mixed_task_Y = torch.cat([mixed_hetero_Y, mixed_hetero_Y + 0.2], dim=0)
mixed_task_pool = torch.cat(
    [mixed_pool, torch.zeros(mixed_pool.shape[0], 1)], dim=-1
)
mixed_task_noise_model = MixedHeteroskedasticMultiTaskGP(
    mixed_task_X, mixed_task_Y, task_feature=2, cat_dims=[1]
)
mixed_task_noise_model.fit_heteroskedastic(iterations=1)
mixed_task_noise_model.eval()
with torch.no_grad():
    mixed_task_scores = (
        mixed_task_noise_model.posterior(mixed_task_pool).mean.reshape(-1)
    )
    mixed_task_noise = mixed_task_noise_model.predicted_noise(mixed_task_pool)
mixed_task_candidate = mixed_task_pool[mixed_task_scores.argmax()].unsqueeze(0)
assert torch.isfinite(mixed_task_noise).all()
print("mixed task candidate:", mixed_task_candidate)
print(
    "mixed task 0 objective:",
    torch.sin(2 * torch.pi * mixed_task_candidate[:, :1])
    + 0.2 * mixed_task_candidate[:, 1:2],
)

fidelity_X = torch.cat(
    [hetero_X, torch.arange(10).remainder(2).to(hetero_X).unsqueeze(-1)],
    dim=-1,
)
fidelity_Y = hetero_Y + 0.15 * (1 - fidelity_X[:, 1:2])
fidelity_noise_model = HeteroskedasticMultiFidelityGP(
    fidelity_X, fidelity_Y, data_fidelities=[1]
)
fidelity_noise_model.fit_heteroskedastic(iterations=1)
fidelity_noise_model.eval()
fidelity_pool = torch.cat(
    [hetero_pool, torch.ones_like(hetero_pool)], dim=-1
)
with torch.no_grad():
    fidelity_scores = fidelity_noise_model.posterior(fidelity_pool).mean.reshape(-1)
    fidelity_noise = fidelity_noise_model.predicted_noise(fidelity_pool)
fidelity_candidate = fidelity_pool[fidelity_scores.argmax()].unsqueeze(0)
assert torch.isfinite(fidelity_noise).all()
print("high-fidelity candidate:", fidelity_candidate)
print("high-fidelity objective:", torch.sin(2 * torch.pi * fidelity_candidate[:, :1]))


## Replicate-noise multi-fidelity candidate proposal

忠実度を含む同一条件の反復測定を集約し、最高忠実度に固定した候補を予測平均で選びます。


In [ ]:
from robotorchan.models import ReplicateNoiseMultiFidelityGP

replicate_fidelity_X = fidelity_X.repeat_interleave(3, dim=0)
replicate_fidelity_Y = (
    torch.sin(2 * torch.pi * replicate_fidelity_X[:, :1])
    + 0.15 * (1 - replicate_fidelity_X[:, 1:2])
    + 0.03 * torch.randn(replicate_fidelity_X.shape[0], 1)
)
replicate_fidelity_model = ReplicateNoiseMultiFidelityGP.from_replicates(
    replicate_fidelity_X, replicate_fidelity_Y, data_fidelities=[1]
)
from botorch.fit import fit_gpytorch_mll

fit_gpytorch_mll(replicate_fidelity_model.make_mll())
replicate_fidelity_model.eval()
with torch.no_grad():
    replicate_fidelity_scores = (
        replicate_fidelity_model.posterior(fidelity_pool).mean.reshape(-1)
    )
replicate_fidelity_candidate = (
    fidelity_pool[replicate_fidelity_scores.argmax()].unsqueeze(0)
)
assert replicate_fidelity_candidate.shape == (1, 2)
assert replicate_fidelity_candidate[0, 1].item() == 1.0
print("replicate multi-fidelity candidate:", replicate_fidelity_candidate)
print(
    "high-fidelity objective:",
    torch.sin(2 * torch.pi * replicate_fidelity_candidate[:, :1]),
)


## Joint variational heteroskedastic GP

平均と対数観測分散を同時に変分学習する2モデルです。既存の `training_loss()` をAdamで最小化し、候補は有限プールの予測平均から選びます。獲得関数最適化ではありません。


In [ ]:
from robotorchan.models import (
    JointHeteroskedasticSingleTaskGP,
    MixedJointHeteroskedasticSingleTaskGP,
)

joint_variants = [
    (
        "JointHeteroskedasticSingleTaskGP",
        JointHeteroskedasticSingleTaskGP(
            hetero_X, hetero_Y, num_inducing=5, num_mc_samples=4
        ),
        hetero_pool,
    ),
    (
        "MixedJointHeteroskedasticSingleTaskGP",
        MixedJointHeteroskedasticSingleTaskGP(
            mixed_hetero_X, mixed_hetero_Y,
            cat_dims=[1], num_inducing=5, num_mc_samples=4,
        ),
        mixed_pool,
    ),
]
for name, joint_model, pool in joint_variants:
    joint_model.train()
    joint_optimizer = torch.optim.Adam(joint_model.parameters(), lr=0.01)
    for _ in range(3):
        joint_optimizer.zero_grad()
        joint_loss = joint_model.training_loss()
        joint_loss.backward()
        joint_optimizer.step()
    joint_model.eval()
    with torch.no_grad():
        joint_scores = joint_model.posterior(pool).mean.reshape(-1)
        joint_noise = joint_model.predicted_noise(pool)
    joint_candidate = pool[joint_scores.argmax()].unsqueeze(0)
    assert torch.isfinite(joint_scores).all()
    assert torch.isfinite(joint_noise).all()
    if joint_candidate.shape[-1] == 1:
        joint_objective = torch.sin(2 * torch.pi * joint_candidate)
    else:
        joint_objective = (
            torch.sin(2 * torch.pi * joint_candidate[:, :1])
            + 0.2 * joint_candidate[:, 1:2]
        )
    print(name, "candidate:", joint_candidate)
    print(name, "evaluated objective:", joint_objective)
